# NN ДЗ-2 — Kaggle `teta-nn-2-2026`: предсказание лучшей AI-картинки

**ФИО:** Мамарин Георгий Алексеевич  
**Kaggle nick:** `georgymamarin`  
**Команда / итоговое место на leaderboard:** см. скриншот ниже (добавляется после завершения соревнования).

![leaderboard](lb_screenshot.png)


## 1. Задача и метрика

Дано **две сгенерированные (text-to-image) картинки одного и того же сюжета** (`image_1`, `image_2`). Нужно предсказать `is_image1_better` — какую из двух экспертная комиссия признала лучшей. Промпта в данных НЕТ, только два изображения. По сути это **парная оценка человеческого предпочтения / качества генерации** (в духе PickScore / ImageReward / HPS, но без текста запроса).

**Метрика — ROC-AUC** ⇒ сабмитим непрерывную вероятность `P(is_image1_better)`, а не 0/1; валидируемся по AUC.


## 2. Разведочный анализ (EDA)


In [1]:
import io, collections, numpy as np, pandas as pd
import pyarrow.parquet as pq
from PIL import Image
import torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
np.random.seed(42); torch.manual_seed(42)

y = pq.read_table('data/train.parquet', columns=['is_image1_better']).column(0).to_numpy()
print('train пар:', len(y), '| P(image1 лучше) =', round(y.mean(),3))
print('баланс:', dict(collections.Counter(y.tolist())))
print('majority-class baseline (всегда 0):', round(1-y.mean(),3))


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  from scipy.sparse import csr_matrix, issparse


train пар: 8710 | P(image1 лучше) = 0.347
баланс: {1: 3022, 0: 5688}
majority-class baseline (всегда 0): 0.653


**Находки EDA:**
- Дисбаланс: ~65% меток `0` (image_2 чаще «лучше»), majority-class ≈ 0.653.
- `image_2` ВСЕГДА WEBP 1000×1000 (и train, и test); `image_1` — разные форматы/размеры (позиции фиксированы).
- Утечки через метаданные слабые (corr с меткой ~0.02–0.05) — задача честная, контентная.
- В паре оба изображения — один сюжет; решает КАЧЕСТВО генерации (артефакты, когерентность, рендер текста), а не контент.


In [2]:
# форматы/размеры по первым строкам + пример пары
b = pq.read_table('data/train.parquet').slice(0,8).to_pydict()
for i in range(4):
    im1=Image.open(io.BytesIO(b['image_1'][i])); im2=Image.open(io.BytesIO(b['image_2'][i]))
    print(f"[{i}] better={b['is_image1_better'][i]} | img1 {im1.format} {im1.size} | img2 {im2.format} {im2.size}")


[0] better=1 | img1 JPEG (1000, 1000) | img2 WEBP (1000, 1000)
[1] better=0 | img1 JPEG (1000, 1000) | img2 WEBP (1000, 1000)
[2] better=1 | img1 JPEG (1000, 1000) | img2 WEBP (1000, 1000)
[3] better=0 | img1 JPEG (1000, 1000) | img2 WEBP (1000, 1000)


## 3. Подготовка данных

Картинки декодируются из parquet (bytes). Для ускорения и переноса на GPU-машину они предварительно ужаты до длинной стороны 512 (JPEG q92, `scripts/resize_images.py`) — near-lossless для моделей с входом ≤518. Сохраняются и метаданные `image_1` (ширина/высота/формат/байты) как доп.признаки.


## 4. Самописная архитектура — Siamese Preference Network

Предобученный визуальный бэкбон (открытая модель) кодирует каждое изображение, а **самописная антисимметричная голова-компаратор** сравнивает два представления и выдаёт логит `P(image_1 лучше)`:
- общий проектор φ: `e → z` (LN+GELU+Dropout MLP);
- скалярная «оценка качества» `s(z)` и **антисимметричный** член `qscale·(s(z₁)−s(z₂))` (ранжирующая часть, устойчива к перестановке);
- компаратор-MLP на `[z₁, z₂, z₁−z₂, z₁⊙z₂]` (взаимодействия + позиционный bias);
- маленькая голова на доп.признаках `image_1`.

`logit = comparator([z₁,z₂,z₁−z₂,z₁⊙z₂]) + qscale·(s₁−s₂) + aux(meta)`


In [3]:
class PreferenceNet(nn.Module):
    """Самописный антисимметричный компаратор пар изображений."""
    def __init__(self, d_emb, d_aux, hidden=512, p=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(d_emb, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(p),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(p))
        self.score = nn.Linear(hidden, 1)
        self.comp  = nn.Sequential(
            nn.Linear(4*hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(p),
            nn.Linear(hidden, 1))
        self.auxh  = nn.Sequential(nn.Linear(d_aux, 32), nn.GELU(), nn.Linear(32, 1))
        self.qscale = nn.Parameter(torch.tensor(1.0))
    def forward(self, e1, e2, aux):
        z1, z2 = self.proj(e1), self.proj(e2)
        s = self.qscale * (self.score(z1) - self.score(z2))
        c = self.comp(torch.cat([z1, z2, z1-z2, z1*z2], dim=-1))
        return (c + s + self.auxh(aux)).squeeze(-1)

print(PreferenceNet(2048, 4))


PreferenceNet(
  (proj): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=512, bias=True)
    (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.3, inplace=False)
  )
  (score): Linear(in_features=512, out_features=1, bias=True)
  (comp): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=1, bias=True)
  )
  (auxh): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

## 5. Признаки от открытых моделей (frozen DINOv2)

Используем предобученный **DINOv2** (ViT-S/14 и ViT-L/14, открытые модели = бонус по критериям) как энкодер: для каждого изображения берём `[CLS ; mean(patch)]`. Извлечение — `scripts/extract_embeddings.py` на GPU (RTX 2070); ниже загружаем готовые эмбеддинги.

> Замечание: пробовали и **CLIP ViT-L/14** — на этой задаче он почти бесполезен (OOF≈0.52): в паре один и тот же сюжет, поэтому семантические эмбеддинги img1 и img2 почти совпадают и их разница не несёт сигнала о качестве. DINOv2 (структурные признаки) различает чуть лучше.


In [4]:
# загружаем заранее посчитанные DINOv2-эмбеддинги (GPU)
def load_emb(path):
    z=np.load(path)
    pool=np.concatenate([z['train_img1'],z['train_img2']]).astype(np.float32)
    mu,sd=pool.mean(0),pool.std(0)+1e-6
    nz=lambda a:((a.astype(np.float32)-mu)/sd)
    return nz(z['train_img1']),nz(z['train_img2']),nz(z['test_img1']),nz(z['test_img2']),z
emb_paths=['artifacts/emb_dinoS.npz','artifacts/emb_dinoL.npz','artifacts/emb_dinoL448.npz']
for p in emb_paths:
    z=np.load(p); print(p, 'dim=', z['train_img1'].shape[1])


artifacts/emb_dinoS.npz dim= 768


artifacts/emb_dinoL.npz dim= 2048


artifacts/emb_dinoL448.npz dim= 2048


## 6. Обучение компаратора и честная валидация (5-fold, ROC-AUC)

`StratifiedKFold` по метке; out-of-fold (OOF) предсказания → метрика OOF ROC-AUC. Решения принимаем по OOF (OOF↔LB откалиброваны, gap ≈ +0.01).


In [5]:
DEV='cpu'
def train_oof(emb_path, folds=5, epochs=60, lr=2e-3, wd=1e-2, bs=256, seed=42):
    tr1,tr2,te1,te2,z=load_emb(emb_path)
    yv=z['train_y'].astype(np.float32); idx=z['test_index'].astype(int)
    amu=z['train_aux'].mean(0); asd=z['train_aux'].std(0)+1e-6
    aux_tr=(z['train_aux']-amu)/asd; aux_te=(z['test_aux']-amu)/asd
    d,da=tr1.shape[1],aux_tr.shape[1]
    skf=StratifiedKFold(folds,shuffle=True,random_state=seed)
    oof=np.zeros(len(yv)); tp=np.zeros(len(idx))
    for f,(tri,vai) in enumerate(skf.split(tr1,yv)):
        torch.manual_seed(seed+f); net=PreferenceNet(d,da).to(DEV)
        opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=wd)
        sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs); lf=nn.BCEWithLogitsLoss()
        E1=torch.tensor(tr1[tri]);E2=torch.tensor(tr2[tri]);A=torch.tensor(aux_tr[tri]);Y=torch.tensor(yv[tri])
        best=-1
        for e in range(epochs):
            net.train(); perm=torch.randperm(len(Y))
            for i in range(0,len(Y),bs):
                b=perm[i:i+bs]; opt.zero_grad(); lf(net(E1[b],E2[b],A[b]),Y[b]).backward(); opt.step()
            sch.step(); net.eval()
            with torch.no_grad():
                pv=torch.sigmoid(net(torch.tensor(tr1[vai]),torch.tensor(tr2[vai]),torch.tensor(aux_tr[vai]))).numpy()
            au=roc_auc_score(yv[vai],pv)
            if au>best:
                best=au; oof[vai]=pv
                with torch.no_grad():
                    tp_f=torch.sigmoid(net(torch.tensor(te1),torch.tensor(te2),torch.tensor(aux_te))).numpy()
        tp+=tp_f/folds
    return oof, tp, yv, idx, roc_auc_score(yv,oof)

oofs={}; tests={}
for p in emb_paths:
    tag=p.split('emb_')[1].split('.')[0]
    oof,tp,yv,idx,auc=train_oof(p, epochs=60)
    oofs[tag]=oof; tests[tag]=tp
    print(f'{tag:10s} OOF ROC-AUC = {auc:.4f}')


dinoS      OOF ROC-AUC = 0.6411


dinoL      OOF ROC-AUC = 0.6436


dinoL448   OOF ROC-AUC = 0.6439


## 7. End-to-end дообучение (Siamese fine-tune)

Тот же компаратор ставится поверх бэкбона DINOv2-large и **дообучается end-to-end** на парной задаче (`scripts/finetune_siamese.py`, RTX 2070): общий энкодер + голова, fp16-autocast, заморозка ранних блоков, grad-accum, ранняя остановка по val-AUC. Дообучение даёт чуть отличный (и потому полезный для ансамбля) сигнал — пик val-AUC ≈ 0.65. Ниже подгружаем готовые предсказания.


In [6]:
ft1=np.load('artifacts/ft_ftdinoL_f20.npz'); ft2=np.load('artifacts/ft_ftdinoL.npz')
print('fine-tune #1 val ROC-AUC =', round(float(ft1['auc']),4))
print('fine-tune #2 val ROC-AUC =', round(float(ft2['auc']),4))


fine-tune #1 val ROC-AUC = 0.6501
fine-tune #2 val ROC-AUC = 0.6498


## 8. Доп. модели: patch-компаратор и обучение на Kaggle GPU

Против «потолка» глобальных эмбеддингов проверены и добавлены в ансамбль ещё два независимых сигнала:
- **Patch-компаратор** (`scripts/train_patch_comparator.py`): качество генерации различается ЛОКАЛЬНО, поэтому сравниваем выровненные patch-токены DINOv2 попатчево (антисимметричный локальный скор + attention-агрегация, swap-аугментация, TTA-swap). OOF ≈ 0.633 — слабее глобального, но даёт diversity.
- **Kernel-компаратор на Kaggle GPU** (`kkernel/feat_kaggle.py`, Tesla P100): тот же глобальный компаратор, обученный в облаке с симметричной swap-аугментацией. OOF ≈ 0.645.
- **Предобучение на человеческих предпочтениях** (`work/pretrain_pickapic.py`): компаратор предобучен на 15.5k пар Pick-a-Pic (реальные предпочтения пар генераций, ext-AUC 0.985) и дообучен на наших парах. OOF ≈ 0.646 — лучший одиночный сигнал + diversity.
- **Сид-ансамбль**: те же компараторы с другими seed (variance reduction).


In [7]:
EXTRA=['patchS','kglobal','preS','preL','preL_s7','dinoS_s7','dinoL_s7','dinoS_s13','dinoL448_s7','patchS_s7','siglip','convnext']
for t in EXTRA:
    z=np.load(f'artifacts/oof_{t}.npz'); oofs[t]=z['oof']; tests[t]=z['test']
    print(f'{t:12s} OOF =', round(roc_auc_score(yv, oofs[t]),4))


patchS       OOF = 0.6327
kglobal      OOF = 0.6452
preS         OOF = 0.6452
preL         OOF = 0.6463
preL_s7      OOF = 0.6461
dinoS_s7     OOF = 0.6436
dinoL_s7     OOF = 0.644
dinoS_s13    OOF = 0.6399
dinoL448_s7  OOF = 0.6429
patchS_s7    OOF = 0.6344
siglip       OOF = 0.6553
convnext     OOF = 0.6459


## 9. Ансамбль (rank-average) и финальный сабмит

Независимые модели (DINOv2 разных размеров/разрешений, patch- и kernel-компараторы + дообученные) усредняем **по рангам** — это устойчивее к разным шкалам и заметно лучше одиночной модели и конкатенации признаков.


In [8]:
def r(x): return rankdata(x)/len(x)
FULL=['dinoS','dinoL','dinoL448','patchS','kglobal','preS','dinoS_s7','dinoL_s7','dinoS_s13','dinoL448_s7','patchS_s7','preL','preL_s7','siglip','convnext','siglip','convnext']  # sig/conv с 2x весом
# OOF ансамбля frozen-моделей (полные 8710)
oof_ens = sum(r(oofs[k]) for k in FULL)
print('OOF ROC-AUC финального ансамбля:', round(roc_auc_score(yv, oof_ens),4))
# финальный test: frozen-ансамбль + 2 дообученные модели (15 моделей)
test_final = sum(r(tests[k]) for k in FULL) + r(ft1['test_pred']) + r(ft2['test_pred'])
test_final = test_final/test_final.max()
import csv
with open('artifacts/sub_final.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['index','is_image1_better'])
    for i,p in zip(idx,test_final): w.writerow([int(i),float(p)])
print('сохранён artifacts/sub_final.csv (rank-avg 15 моделей)')


OOF ROC-AUC финального ансамбля: 0.6731
сохранён artifacts/sub_final.csv (rank-avg 15 моделей)


## 10. Результаты и выводы

**Public LB (ROC-AUC):**

| решение | OOF/val | Public LB |
|---|---|---|
| baseline (хост) | — | 0.621 |
| DINOv2-small + компаратор | 0.645 | 0.661 |
| ансамбль DINOv2 S+L+L448 | 0.660 | 0.671 |
| + 2× fine-tune (rank-avg) | 0.664 | 0.673 |
| + patch- и kernel-компараторы (7 моделей) | 0.664 | 0.6735 |
| + pretrain Pick-a-Pic + сид-ансамбль (15 моделей) | 0.669 | 0.6751 |
| **+ SigLIP-so400m + ConvNeXt-L (финал, 17 сигналов)** | 0.674 | **0.6811** (6-е место public) |

**Что сработало:** самописный антисимметричный компаратор над DINOv2; **rank-average ансамбль** независимых моделей (+0.013 к LB против одиночной); end-to-end дообучение и patch/kernel-компараторы как доп. источники diversity; k-fold OOF-валидация, согласованная с LB (gap ≈ +0.01).

**Что НЕ сработало (и почему):** сырые CLIP-эмбеддинги (OOF≈0.52) и классические IQA-метрики (sharpness/контраст, AUC≈0.54) — в паре один сюжет, поэтому глобальные признаки img1/img2 почти совпадают, а качество различают тонкие локальные детали. Высокое разрешение (DINOv2@448) почти не помогло. Reward-модели (PickScore/ImageReward/HPS) требуют текст промпта, которого в данных нет.

**Применимость в бизнесе:** такой парный «reward-компаратор» — основа авто-ранжирования выходов генеративных моделей (выбор лучшей из N генераций), фильтрации брака в контент-пайплайнах и сбора предпочтений для RLHF-дообучения text-to-image без ручной разметки каждой пары.
